# Free Steak Dinner VOR Model — 2026

Draft board for Yahoo league 410984: 12 teams, half-PPR, 6-pt passing TDs,
-1 INT, 0.5 per rushing first down, QB/2RB/3WR/TE/W-R-T/K/DEF, 17 roster slots.

**Run order matters.** Runtime -> Restart runtime -> Run all. Cell 9 downloads
~5 seasons of nflverse data and is slow; everything else is fast.

### What this does that public rankings don't
1. Scores every player from raw projected stats using **your exact league
   settings** — including rushing first downs (0.5) and 40+ yard receptions,
   which no public half-PPR ranking prices in.
2. Derives replacement level from **your roster structure** (12 teams, 3 WR,
   flex split 75/25 to RB) rather than a generic assumption: QB13 / RB34 /
   WR40 / TE13.
3. Ranks K and DST by **draft reality** — exactly 12 of each are drafted, in
   the last rounds — instead of letting season-long VOR put a defense 66th.
4. Shows **availability risk** (VOR_ADJ) from empirical games-played rates by
   position and age, because the projections assume everyone plays 18 games.
5. Puts **ROOM+** on every row: how much later the room (drafting off the
   FantasyPros half-PPR cheatsheet) will take a player than your board says.

### Known limits
- Single projection source (Sleeper). Aggregated projections are modestly
  more accurate; revisit in the offseason with a backtest, not on draft week.
- Not scored (no source projects them): 40+ yard runs/completions, 40+ TD
  bonuses, return TDs, pick-six penalty.
- K/DST use Sleeper's own scoring; their VOR spread is small enough that the
  approximation costs nothing.



In [ ]:
# ============================================================
# CELL 1 — CONFIG + SHARED HELPERS
# Everything tunable lives here. Nothing below needs editing.
# ============================================================
import json
import re
from datetime import date

import numpy as np
import pandas as pd
import requests

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
POSITIONS = ["QB", "RB", "WR", "TE", "K", "DST"]
NFLVERSE = "https://github.com/nflverse/nflverse-data/releases/download"


def _norm(n):
    """Normalize a player name for cross-source matching. Sleeper omits
    suffixes (Jr./Sr./II/III); FantasyPros and nflverse include them."""
    n = str(n).lower().replace(".", "").replace("'", "").replace("-", " ")
    n = re.sub(r"\s+(jr|sr|ii|iii|iv|v)$", "", n)
    return re.sub(r"\s+", " ", n).strip()


def load_first(candidates, label):
    """Try candidate URLs in order; return the first that loads. nflverse
    renames release assets between years, so we probe rather than assume."""
    for url in candidates:
        try:
            df = (pd.read_parquet(url) if url.endswith(".parquet")
                  else pd.read_csv(url, low_memory=False))
            print(f"OK  {label}: {url.split('/')[-1]}  ({len(df):,} rows)")
            return df
        except Exception as e:
            print(f"    no: {url.split('/')[-1]}  ({type(e).__name__})")
    raise RuntimeError(f"could not load {label}")


# --- Scoring -------------------------------------------------------------
SCORING = {
    "pass_yd": 0.04,       # 25 passing yards per point
    "pass_td": 6.0,        # league override (Yahoo default is 4)
    "int": -1.0,
    "pass_40p": 1.0,       # not projected by Sleeper -> contributes 0
    "rush_yd": 0.1,
    "rush_td": 6.0,
    "rush_fd": 0.5,        # rushing first downs — a real projected stat
    "rush_40p": 2.0,       # not projected by Sleeper -> contributes 0
    "rec": 0.5,            # HALF-PPR
    "rec_yd": 0.1,
    "rec_td": 6.0,
    "rec_40p": 1.0,
    "fumble": -2.0,
    "two_pt": 2.0,
}

# Fallback only — used if Sleeper's rush_fd is ever unpopulated. Fit on
# 2022-25 nflverse actuals; your original 0.08/0.05 guesses were within 4%.
RUSH_1D_RATES = {"QB": 0.078, "RB": 0.052, "WR": 0.052, "TE": 0.052,
                 "K": 0.0, "DST": 0.0}

# --- Roster (drives replacement levels) ----------------------------------
ROSTER = {
    "teams": 12,
    "slots": {"QB": 1, "RB": 2, "WR": 3, "TE": 1, "K": 1, "DST": 1},
    "flex": 1,                                     # the W/R/T spot
    "flex_split": {"RB": 0.75, "WR": 0.25, "TE": 0.0},   # how your league fills it
}
REPLACEMENT_OVERRIDE = {}      # e.g. {"QB": 15}; normally leave empty

# --- Draft sheet shape ---------------------------------------------------
SHEET_SIZE = 200
LATE_POS = {"DST": 12, "K": 12}    # exactly one each per team, taken last


In [ ]:
# ============================================================
# CELL 2 — SLEEPER PROJECTIONS API
#
# FantasyPros moved projections and ADP to client-side rendering in 2026:
# their HTML serves 10 rows per position and the ADP table is gone. Sleeper's
# public endpoint returns full stat lines AND half-PPR ADP as JSON — no auth,
# no HTML parsing, and no cross-source name matching, since stats and ADP
# arrive in the same record. It also projects rushing first downs directly.
# ============================================================
SLEEPER_SEASON = "2026"
SLEEPER_URL = f"https://api.sleeper.app/projections/nfl/{SLEEPER_SEASON}"

STAT_MAP = {
    "PASS_YD": "pass_yd", "PASS_TD": "pass_td", "INT": "pass_int",
    "PASS_40P": "pass_40p", "PASS_2PT": "pass_2pt",
    "RUSH_ATT": "rush_att", "RUSH_YD": "rush_yd", "RUSH_TD": "rush_td",
    "RUSH_FD": "rush_fd", "RUSH_40P": "rush_40p", "RUSH_2PT": "rush_2pt",
    "REC": "rec", "REC_YD": "rec_yd", "REC_TD": "rec_td",
    "REC_FD": "rec_fd", "REC_40P": "rec_40p", "REC_2PT": "rec_2pt",
    "FL": "fum_lost", "GP": "gp",
    "ADP_HALF": "adp_half_ppr", "SLEEPER_PTS_HALF": "pts_half_ppr",
}
SLEEPER_POS_MAP = {"DEF": "DST"}
ADP_UNDRAFTED = 900.0              # Sleeper uses 999.0 as "no ADP"


def fetch_sleeper():
    params = [("season_type", "regular"), ("order_by", "adp_half_ppr")]
    for p in ("QB", "RB", "WR", "TE", "K", "DEF"):
        params.append(("position[]", p))
    r = requests.get(SLEEPER_URL, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()
    if not isinstance(data, list) or not data:
        raise RuntimeError(f"Sleeper returned unexpected payload: {type(data)}")
    return data


def sleeper_to_frame(records):
    """Flatten Sleeper records into this notebook's canonical schema."""
    rows, seen_keys = [], {}
    for rec in records:
        p = rec.get("player") or {}
        stats = rec.get("stats") or {}
        pos = p.get("position") or (p.get("fantasy_positions") or [None])[0]
        if not pos:
            continue
        pos = SLEEPER_POS_MAP.get(pos, pos)
        if pos not in POSITIONS:
            continue
        name = (p.get("full_name")
                or " ".join(filter(None, [p.get("first_name"), p.get("last_name")]))
                or "").strip()
        if not name:
            continue
        row = {"Player": name, "Team": (p.get("team") or "FA"), "POS": pos,
               "SLEEPER_ID": rec.get("player_id") or p.get("player_id")}
        for col, key in STAT_MAP.items():
            row[col] = float(stats.get(key) or 0.0)
        rows.append(row)
        seen_keys.setdefault(pos, set()).update(stats.keys())

    df = pd.DataFrame(rows)
    df.loc[df["ADP_HALF"] >= ADP_UNDRAFTED, "ADP_HALF"] = np.nan
    df.loc[df["ADP_HALF"] == 0.0, "ADP_HALF"] = np.nan
    return df, seen_keys


def discovery_report(df, seen_keys):
    """Which mapped fields actually arrived. Read after a season rollover:
    a renamed Sleeper field shows up here instead of silently scoring 0."""
    print("\n--- DISCOVERY REPORT ---")
    mapped = set(STAT_MAP.values())
    for pos in POSITIONS:
        keys = seen_keys.get(pos, set())
        if not keys:
            print(f"{pos}: NO RECORDS")
            continue
        missing = sorted(k for k in mapped if k not in keys)
        print(f"{pos}: {len(df[df['POS'] == pos])} players, {len(keys)} stat keys")
        if missing:
            print(f"   mapped-but-absent: {missing}")


In [ ]:
# ============================================================
# CELL 3 — FETCH + SNAPSHOT
# Saves dated CSVs so any board is reproducible later — and so you have
# preseason-frozen data to backtest against in January.
# ============================================================
USE_FANTASYPROS_ECR = True      # the cheatsheet page is what your room drafts off

records = fetch_sleeper()
print(f"Sleeper: {len(records)} raw records")
proj, seen_keys = sleeper_to_frame(records)
print(f"usable players: {len(proj)}")
discovery_report(proj, seen_keys)

# --- ADP comes from the same payload: no name matching needed ---
adp = (proj.loc[proj["ADP_HALF"].notna(), ["Player", "Team", "POS", "ADP_HALF"]]
       .sort_values("ADP_HALF").reset_index(drop=True))
adp["ADP_RANK"] = adp.index + 1
adp = adp.rename(columns={"ADP_HALF": "AVG"})
print(f"\nplayers with half-PPR ADP: {len(adp)}")

# --- FantasyPros consensus (the cheatsheet your league drafts from) ---
if USE_FANTASYPROS_ECR:
    def extract_js_json(html, varname):
        """Pull `varname = {...};` out of page JS with brace matching —
        regex breaks on braces inside strings."""
        i = html.find(varname)
        if i == -1:
            raise ValueError(f"{varname} not found")
        s = html.find("=", i) + 1
        while html[s] in " \t\r\n":
            s += 1
        oc = html[s]
        cc = "}" if oc == "{" else "]"
        depth = 0; j = s; in_str = False; esc = False
        while j < len(html):
            c = html[j]
            if in_str:
                if esc: esc = False
                elif c == "\\": esc = True
                elif c == '"': in_str = False
            else:
                if c == '"': in_str = True
                elif c == oc: depth += 1
                elif c == cc:
                    depth -= 1
                    if depth == 0:
                        return json.loads(html[s:j + 1])
            j += 1
        raise ValueError(f"unterminated {varname}")

    fp_html = requests.get(
        "https://www.fantasypros.com/nfl/rankings/half-point-ppr-cheatsheets.php",
        headers=HEADERS, timeout=30).text
    ecr = extract_js_json(fp_html, "ecrData")
    fp = pd.DataFrame([{
        "Player": p["player_name"],
        "POS": SLEEPER_POS_MAP.get(p["player_position_id"], p["player_position_id"]),
        "ECR": p.get("rank_ecr"),
        "FP_TIER": p.get("tier"),
        "Bye": p.get("player_bye_week"),
        # expert disagreement = a free uncertainty signal
        "ECR_SD": pd.to_numeric(p.get("rank_std"), errors="coerce"),
        "ECR_BEST": pd.to_numeric(p.get("rank_min"), errors="coerce"),
        "ECR_WORST": pd.to_numeric(p.get("rank_max"), errors="coerce"),
    } for p in ecr["players"]])
    print(f"FantasyPros ECR: {len(fp)} players (as of {ecr.get('last_updated')})")
else:
    fp = None

stamp = date.today().isoformat()
proj.to_csv(f"projections_{stamp}.csv", index=False)
adp.to_csv(f"adp_{stamp}.csv", index=False)
print(f"snapshots saved: projections_{stamp}.csv, adp_{stamp}.csv")

# --- is rush_fd populated? decide once, on players with real volume ---
_r = proj[(proj["POS"].isin(["QB", "RB"])) & (proj["RUSH_ATT"] >= 20)]
share = (_r["RUSH_FD"] > 0).mean()
RUSH_FD_OK = bool(share > 0.8)
print(f"rush_fd populated for {share:.0%} of {len(_r)} QB/RB with 20+ carries "
      f"-> RUSH_FD_OK={RUSH_FD_OK}")


In [ ]:
# ============================================================
# CELL 4 — PLAYER METADATA + AGE
# Sleeper has no age field, so age comes from nflverse. The join is
# position-aware because name-only matching put the wrong Josh Allen's
# birthday on the quarterback.
# ============================================================
META_FIELDS = ["years_exp", "injury_status", "injury_body_part",
               "injury_notes", "team_changed_at"]

meta = []
for rec in records:
    p = rec.get("player") or {}
    row = {"SLEEPER_ID": rec.get("player_id") or p.get("player_id")}
    for f in META_FIELDS:
        row[f.upper()] = p.get(f)
    meta.append(row)

meta = pd.DataFrame(meta).drop_duplicates("SLEEPER_ID")
proj = proj.drop(columns=[f.upper() for f in META_FIELDS], errors="ignore")
proj = proj.merge(meta, on="SLEEPER_ID", how="left")
proj["YEARS_EXP"] = pd.to_numeric(proj["YEARS_EXP"], errors="coerce")
print(f"years_exp populated for {proj['YEARS_EXP'].notna().mean():.0%} of players")

ros = load_first([
    f"{NFLVERSE}/players/players.parquet",
    f"{NFLVERSE}/rosters/roster_2025.parquet",
    f"{NFLVERSE}/rosters/rosters_2025.parquet",
], "player birth dates")

namecol = next((c for c in ("display_name", "player_name", "full_name")
                if c in ros.columns), None)
birthcol = next((c for c in ("birth_date", "birthdate") if c in ros.columns), None)
poscol = next((c for c in ("position", "position_group") if c in ros.columns), None)
if namecol is None or birthcol is None:
    raise RuntimeError(f"unexpected schema: {list(ros.columns)[:40]}")

cols = [namecol, birthcol] + ([poscol] if poscol else [])
ros = ros[cols].dropna(subset=[namecol, birthcol]).copy()
ros[birthcol] = pd.to_datetime(ros[birthcol], errors="coerce")
ros["AGE"] = ((pd.Timestamp("2026-09-01") - ros[birthcol]).dt.days / 365.25).round(1)
ros = ros[(ros["AGE"] > 18) & (ros["AGE"] < 50)]
ros["_k"] = ros[namecol].map(_norm)

proj["_k"] = proj["Player"].map(_norm)
proj = proj.drop(columns=["AGE"], errors="ignore")

if poscol:
    ros["_kp"] = ros["_k"] + "|" + ros[poscol].astype(str).str.upper()
    proj["_kp"] = proj["_k"] + "|" + proj["POS"]
    proj["AGE"] = ros.drop_duplicates("_kp").set_index("_kp")["AGE"].reindex(proj["_kp"]).values
    miss = proj["AGE"].isna()
    proj.loc[miss, "AGE"] = (ros.drop_duplicates("_k").set_index("_k")["AGE"]
                             .reindex(proj.loc[miss, "_k"]).values)
    proj = proj.drop(columns=["_kp"])
else:
    proj["AGE"] = ros.drop_duplicates("_k").set_index("_k")["AGE"].reindex(proj["_k"]).values

drafted = proj[proj["ADP_HALF"].notna()]
print(f"\nage matched for {proj['AGE'].notna().mean():.0%} of all players, "
      f"{drafted['AGE'].notna().mean():.0%} of players with ADP")
print(drafted.groupby("POS")["AGE"].median().round(1))


In [ ]:
# ============================================================
# CELL 5 — CUSTOM SCORING
# Skill positions are scored from raw stats with your exact settings.
# K/DST use Sleeper's own points: the payload omits the components your
# league's rules need (no FG total, no points-allowed), and the VOR
# spread at those positions is too small for the difference to matter.
# ============================================================
def score_row(row):
    def g(col):
        """Missing or NaN -> 0.0. Without the NaN guard one blank field
        turns PROJ_PTS into NaN and drops the player from the ordering."""
        v = row.get(col, 0.0)
        try:
            v = float(v)
        except (TypeError, ValueError):
            return 0.0
        return 0.0 if pd.isna(v) else v

    if row["POS"] in ("K", "DST"):
        return g("SLEEPER_PTS_HALF")

    rush_fd = (g("RUSH_FD") if RUSH_FD_OK
               else g("RUSH_YD") * RUSH_1D_RATES.get(row["POS"], 0.0))

    return (
        g("PASS_YD") * SCORING["pass_yd"]
        + g("PASS_TD") * SCORING["pass_td"]
        + g("INT") * SCORING["int"]
        + g("PASS_40P") * SCORING["pass_40p"]
        + g("RUSH_YD") * SCORING["rush_yd"]
        + g("RUSH_TD") * SCORING["rush_td"]
        + rush_fd * SCORING["rush_fd"]
        + g("RUSH_40P") * SCORING["rush_40p"]
        + g("REC") * SCORING["rec"]
        + g("REC_YD") * SCORING["rec_yd"]
        + g("REC_TD") * SCORING["rec_td"]
        + g("REC_40P") * SCORING["rec_40p"]
        + g("FL") * SCORING["fumble"]
        + (g("PASS_2PT") + g("RUSH_2PT") + g("REC_2PT")) * SCORING["two_pt"]
    )


proj["PROJ_PTS"] = proj.apply(score_row, axis=1).round(1)

for pos in POSITIONS:
    top = (proj[proj["POS"] == pos].nlargest(3, "PROJ_PTS")[["Player", "PROJ_PTS"]]
           .to_string(index=False, header=False).replace("\n", " | "))
    print(f"{pos:>4}: {top}")


In [ ]:
# ============================================================
# CELL 6 — REPLACEMENT LEVELS, VOR, DRAFT-ORDER RANKING
#
# Replacement = last starter + 1, from your roster:
#   QB13 / RB34 / WR40 / TE13  (RB/WR include their share of the flex)
#
# K and DST get a different treatment. Exactly 12 of each are drafted —
# one per team — and they go in the final rounds. Season-long VOR can't
# fairly rank a position you stream weekly against one you hold all year,
# so we don't ask it to: skill players fill the first 176 slots, then the
# 12 defenses and 12 kickers fill out the top 200 (picks 177-200, i.e.
# rounds 15-17). Kickers and defenses past the 12th are dropped.
# ============================================================
def replacement_ranks():
    out = {}
    for pos in POSITIONS:
        starters = ROSTER["teams"] * ROSTER["slots"].get(pos, 0)
        flex = (round(ROSTER["teams"] * ROSTER["flex"] * ROSTER["flex_split"].get(pos, 0.0))
                if pos in ("RB", "WR", "TE") else 0)
        out[pos] = REPLACEMENT_OVERRIDE.get(pos, starters + flex + 1)
    return out


repl_rank = replacement_ranks()
repl_pts = {}
for pos in POSITIONS:
    pool = proj.loc[proj["POS"] == pos].sort_values("PROJ_PTS", ascending=False)
    if len(pool) == 0:
        repl_pts[pos] = 0.0
        continue
    idx = min(repl_rank[pos] - 1, len(pool) - 1)
    repl_pts[pos] = float(pool["PROJ_PTS"].iloc[idx])

print("replacement levels:",
      {p: f"{p}{repl_rank[p]} = {repl_pts[p]:.1f} pts" for p in POSITIONS})

proj["VOR"] = proj.apply(lambda r: r["PROJ_PTS"] - repl_pts[r["POS"]], axis=1).round(1)
proj["POS_RANK"] = proj.groupby("POS")["PROJ_PTS"].rank(ascending=False,
                                                        method="first").astype(int)

CUT = SHEET_SIZE - sum(LATE_POS.values())          # 176
skill = proj[~proj["POS"].isin(LATE_POS)].sort_values("VOR", ascending=False)
late = pd.concat([proj[proj["POS"] == p].nsmallest(n, "POS_RANK")
                  for p, n in LATE_POS.items()]).sort_values(["POS_RANK", "POS"])

proj = pd.concat([skill.iloc[:CUT], late, skill.iloc[CUT:]], ignore_index=True)
proj["VOR_RANK"] = proj.index + 1

head = proj.head(SHEET_SIZE)
print(f"top {SHEET_SIZE}: {(~head['POS'].isin(LATE_POS)).sum()} skill + "
      f"{(head['POS'] == 'DST').sum()} DST + {(head['POS'] == 'K').sum()} K")


In [ ]:
# ============================================================
# CELL 7 — TIERS (gap-based, per position)
# A new tier starts wherever the drop to the next player is unusually
# large. On draft day the count of players left in the current tier is
# what should drive the pick, not raw rank order.
# ============================================================
def assign_tiers(points_desc):
    n = len(points_desc)
    if n == 0:
        return []
    gaps = [points_desc[i - 1] - points_desc[i] for i in range(1, min(n, 40))]
    mean_gap = sum(gaps) / len(gaps) if gaps else 0.0
    threshold = max(mean_gap * 1.25, 4.0)
    tiers, tier = [1], 1
    for i in range(1, n):
        if points_desc[i - 1] - points_desc[i] > threshold:
            tier += 1
        tiers.append(tier)
    return tiers


proj["TIER"] = 0
for pos in POSITIONS:
    pool = proj.loc[proj["POS"] == pos].sort_values("PROJ_PTS", ascending=False)
    proj.loc[pool.index, "TIER"] = assign_tiers(pool["PROJ_PTS"].tolist())


In [ ]:
# ============================================================
# CELL 8 — MERGE ADP + FANTASYPROS ECR
# Sleeper omits name suffixes; FantasyPros includes them, so both sides
# match on a normalized key. Two passes: name+position, then name-only
# for hybrid players the two sources classify differently.
# ============================================================
NAME_ALIASES = {
    # normalized Sleeper name : normalized FantasyPros name
    # "hollywood brown": "marquise brown",
}

board = proj.merge(adp[["Player", "POS", "ADP_RANK", "AVG"]],
                   on=["Player", "POS"], how="left")

FP_COLS = ["ECR", "FP_TIER", "Bye", "ECR_SD", "ECR_BEST", "ECR_WORST"]

if fp is not None:
    board["_k"] = board["Player"].map(_norm).replace(NAME_ALIASES)
    fp = fp.copy()
    fp["_k"] = fp["Player"].map(_norm)

    fp_kp = fp.dropna(subset=["_k"]).drop_duplicates(["_k", "POS"]).set_index(["_k", "POS"])
    fp_k = fp.dropna(subset=["_k"]).drop_duplicates("_k").set_index("_k")

    idx = pd.MultiIndex.from_arrays([board["_k"], board["POS"]])
    for c in FP_COLS:
        board[c] = fp_kp[c].reindex(idx).values
    miss = board["ECR"].isna()
    for c in FP_COLS:
        board.loc[miss, c] = fp_k[c].reindex(board.loc[miss, "_k"]).values

    print(f"ECR matched: {int(board['ECR'].notna().sum())} of {len(board)} players")
    still = board[board["ADP_RANK"].notna() & (board["ADP_RANK"] <= 170)
                  & board["ECR"].isna()]
    if len(still):
        print(f"\n[!] {len(still)} top-170 players missing ECR "
              f"(add to NAME_ALIASES if these are real):")
        print(still[["Player", "POS", "Team", "AGE", "ADP_RANK"]]
              .sort_values("ADP_RANK").to_string(index=False))
    else:
        print("all top-170 ADP players matched an ECR")
else:
    for c in FP_COLS:
        board[c] = np.nan

# VALUE = vs Sleeper ADP (sharp drafters). ROOM+ = vs FantasyPros ECR,
# which is the cheatsheet your league actually drafts from — positive
# means the room takes him later than your board says he's worth.
board["VALUE"] = (board["ADP_RANK"] - board["VOR_RANK"]).round(0)
board["ROOM+"] = (board["ECR"] - board["VOR_RANK"]).round(0)
board["ECR_RANGE"] = (board["ECR_WORST"] - board["ECR_BEST"]).round(0)


In [ ]:
# ============================================================
# CELL 9 — EMPIRICAL AVAILABILITY  (slow: downloads ~5 seasons)
#
# The projections give everyone a full 18 games, which is knowably false.
# Two traps in measuring this properly:
#   POPULATION — nflverse weekly data includes every fringe player, so a
#     raw positional mean measures "games a random NFL body plays" (~10 of
#     17), not "games a draftable starter plays" (~13-16).
#   SELECTION — the cohort must be defined EX-ANTE. Filtering on the same
#     season's production drops the Week-1 ACL tear, which is the outcome
#     we're trying to measure. So the cohort for season Y is the top-N at
#     each position in Y-1, and players who missed all of Y count as zeros.
#
# `games` conflates injury with losing your job — intended, since a benched
# starter is equally useless and VOR_ADJ backfills either from waivers.
# ============================================================
SEASONS = [2021, 2022, 2023, 2024, 2025]        # 17-game era only

wk = []
for y in SEASONS:
    try:
        d = load_first([
            f"{NFLVERSE}/player_stats/player_stats_{y}.parquet",
            f"{NFLVERSE}/stats_player/stats_player_week_{y}.parquet",
        ], f"weekly {y}")
        d["season"] = y
        wk.append(d)
    except Exception as e:
        print(f"skip {y}: {e}")
wk = pd.concat(wk, ignore_index=True)

pcol = "position" if "position" in wk.columns else "position_group"
ncol = "player_display_name" if "player_display_name" in wk.columns else "player_name"
ptscol = "fantasy_points_ppr" if "fantasy_points_ppr" in wk.columns else "fantasy_points"
SKILL = ["QB", "RB", "WR", "TE"]

gp = (wk[wk[pcol].isin(SKILL)]
      .groupby([ncol, pcol, "season"])["week"].nunique().reset_index(name="games"))

season_pts = (wk[wk[pcol].isin(SKILL)]
              .groupby([ncol, pcol, "season"], as_index=False)[ptscol].sum())
season_pts["pos_rank"] = (season_pts.groupby([pcol, "season"])[ptscol]
                          .rank(ascending=False, method="first"))

DRAFT_BUFFER = 1.5      # replacement rank x this = the draftable pool
n_by_pos = {p: max(int(repl_rank[p] * DRAFT_BUFFER), 12) for p in SKILL}
cohort = season_pts[season_pts["pos_rank"] <= season_pts[pcol].map(n_by_pos)].copy()
cohort["season"] += 1                                   # carry forward one year
cohort = cohort[cohort["season"].isin(SEASONS)][[ncol, pcol, "season"]]
cohort = cohort.merge(gp[[ncol, pcol, "season", "games"]],
                      on=[ncol, pcol, "season"], how="left")
cohort["games"] = cohort["games"].fillna(0.0)
cohort["avail"] = cohort["games"] / 17.0
cohort["_k"] = cohort[ncol].map(_norm)

pl = load_first([f"{NFLVERSE}/players/players.parquet"], "players")
pn = next(c for c in ("display_name", "player_name", "full_name") if c in pl.columns)
pl = pl[[pn, "birth_date"]].dropna().copy()
pl["birth_date"] = pd.to_datetime(pl["birth_date"], errors="coerce")
pl["_k"] = pl[pn].map(_norm)
cohort = cohort.merge(pl.drop_duplicates("_k")[["_k", "birth_date"]], on="_k", how="left")
cohort["age"] = ((pd.to_datetime(cohort["season"].astype(str) + "-09-01")
                  - cohort["birth_date"]).dt.days / 365.25)

# three bands, not six: finer buckets leave the 30+ RB cell too thin to
# use, which erases the age effect exactly where it matters most
BUCKETS = [(0, 26, "<26"), (26, 30, "26-30"), (30, 60, "30+")]
MIN_COUNT = 15
cohort["bucket"] = pd.cut(cohort["age"], [b[0] for b in BUCKETS] + [60],
                          labels=[b[2] for b in BUCKETS], right=False)

print("\nGAMES PLAYED BY DRAFTABLE STARTERS (of 17):")
print(cohort.groupby(pcol)["games"].describe()[["count", "mean", "50%", "min"]].round(1))

tbl = (cohort.dropna(subset=["bucket"])
       .groupby([pcol, "bucket"], observed=True)["avail"]
       .agg(["mean", "count"]).round(3))
print("\nAVAILABILITY BY POSITION AND AGE:")
print(tbl)

AVAIL = {(p, b): r["mean"] for (p, b), r in tbl.iterrows() if r["count"] >= MIN_COUNT}
POS_MEAN = cohort.groupby(pcol)["avail"].mean().to_dict()
print(f"\nusable buckets: {len(AVAIL)} of {len(tbl)}")
print("position means:", {k: round(v, 3) for k, v in POS_MEAN.items()})


In [ ]:
# ============================================================
# CELL 10 — AVAILABILITY-ADJUSTED VOR  (display only)
#
# Missed games are backfilled from waivers at replacement level, which
# contributes zero VOR by definition. Expected value from the roster spot
# is G*(p - r), and raw VOR is 18*(p - r), so:
#       VOR_ADJ = VOR * (expected_games / 18)
# Availability scales VOR, not points. VOR stays the ranking; VOR_ADJ is
# a second read on who is carrying risk.
# ============================================================
def _bucket(a):
    if pd.isna(a):
        return None
    for lo, hi, lab in BUCKETS:
        if lo <= a < hi:
            return lab
    return None


def _avail(pos, age):
    if pos in ("K", "DST"):
        return 1.0                      # streamed weekly; no availability drag
    b = _bucket(age)
    if b is not None and (pos, b) in AVAIL:
        return AVAIL[(pos, b)]
    return POS_MEAN.get(pos, 0.88)


board["EXP_AVAIL"] = [_avail(p, a) for p, a in zip(board["POS"], board["AGE"])]
board["EXP_GAMES"] = (board["EXP_AVAIL"] * 18).round(1)
board["VOR_ADJ"] = (board["VOR"] * board["EXP_AVAIL"]).round(1)

last = (gp[gp["season"] == 2025]
        .assign(_k=lambda d: d[ncol].map(_norm))
        .drop_duplicates("_k").set_index("_k")["games"])
board["GP_2025"] = last.reindex(board["_k"]).values

mv = board[(board["ADP_RANK"] <= 170) & (board["VOR"] > 0)].copy()
mv["ADJ_RANK"] = mv["VOR_ADJ"].rank(ascending=False, method="first")
mv["DROP"] = (mv["ADJ_RANK"] - mv["VOR_RANK"]).round(0)
out = mv.nlargest(15, "DROP")[["Player", "POS", "AGE", "VOR", "VOR_ADJ",
                               "EXP_GAMES", "GP_2025", "DROP"]].copy()
out["AGE"] = out["AGE"].round(0).astype("Int64")
print("BIGGEST AVAILABILITY DISCOUNTS (fall furthest when risk is priced in):")
print(out.to_string(index=False))


In [ ]:
# ============================================================
# CELL 11 — ON-SCREEN DRAFT SHEET
# VOR is the ranking. VOR_ADJ / EXP_GAMES are the availability read.
# ============================================================
SHEET_COLS = ["VOR_RANK", "Player", "POS", "POS_RANK", "Team", "AGE", "Bye", "TIER",
              "PROJ_PTS", "VOR", "VOR_ADJ", "EXP_GAMES",
              "ADP_RANK", "VALUE", "ECR", "ROOM+", "ECR_RANGE"]
sheet = board[[c for c in SHEET_COLS if c in board.columns]].copy()
if "AGE" in sheet.columns:
    sheet["AGE"] = sheet["AGE"].round(0).astype("Int64")

pd.set_option("display.max_rows", 250)
pd.set_option("display.width", 220)
sheet.head(200)


In [ ]:
# ============================================================
# CELL 12 — explain(): why is this player ranked here?
# Decomposes projected points into scoring categories and shows the
# context the scoring can't see (age, experience, injury, team change,
# workload).
# ============================================================
def explain(name):
    src = board if "board" in globals() else proj
    m = src[src["Player"].str.contains(name, case=False, na=False)]
    if m.empty:
        print(f"no match for '{name}'")
        return
    r = m.iloc[0]

    def g(col):
        v = r.get(col, 0.0)
        try:
            v = float(v)
        except (TypeError, ValueError):
            return 0.0
        return 0.0 if pd.isna(v) else v

    def v(col):
        x = r.get(col) if col in r.index else None
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return None
        return x

    adp_txt = (f"ADP {v('ADP_RANK'):.0f}" if v("ADP_RANK") is not None else "no ADP")
    print(f"{r['Player']}  ({r['POS']}{r['POS_RANK']}, {r['Team']})")
    print(f"  {r['PROJ_PTS']:.1f} pts | VOR {r['VOR']:.1f} | overall #{r['VOR_RANK']} "
          f"| tier {r['TIER']} | {adp_txt}"
          + (f" | FP ECR {v('ECR'):.0f}" if v("ECR") is not None else ""))

    bits = []
    if v("AGE") is not None:
        bits.append(f"age {float(v('AGE')):.0f}")
    if v("YEARS_EXP") is not None:
        e = int(float(v("YEARS_EXP")))
        bits.append("ROOKIE" if e == 0 else f"{e}y exp")
    if v("INJURY_BODY_PART"):
        bits.append(f"injury: {v('INJURY_STATUS') or 'listed'} ({v('INJURY_BODY_PART')})")
    elif v("INJURY_STATUS"):
        bits.append(f"injury: {v('INJURY_STATUS')}")
    if v("TEAM_CHANGED_AT"):
        bits.append(f"changed team {str(v('TEAM_CHANGED_AT'))[:10]}")
    bits.append(f"{g('GP'):.0f} games projected")
    if v("EXP_GAMES") is not None:
        bits.append(f"{float(v('EXP_GAMES')):.1f} expected")
    if r["POS"] in ("RB", "WR", "TE"):
        bits.append(f"{g('RUSH_ATT') + g('REC'):.0f} touches")
    print("  " + " | ".join(bits))

    if v("ECR_RANGE") is not None:
        print(f"  expert range: {v('ECR_BEST'):.0f}-{v('ECR_WORST'):.0f} "
              f"(sd {v('ECR_SD'):.1f}) — wide means the experts disagree")
    if v("INJURY_NOTES"):
        print(f"  note: {str(v('INJURY_NOTES'))[:160]}")

    if r["POS"] in ("K", "DST"):
        print("  (scored with Sleeper's own points; ranked by draft reality, "
              "not season-long VOR)")
        return

    sl = g("SLEEPER_PTS_HALF")
    if sl > 0:
        print(f"  league premium: {r['PROJ_PTS'] - sl:+.1f} pts vs standard "
              f"half-PPR ({sl:.1f}) — this is your edge over the draft room")

    age = float(v("AGE")) if v("AGE") is not None else None
    if r["POS"] == "RB" and age is not None and age >= 28:
        print("  [!] RB 28+ and the projection assumes a full season")
    if r["POS"] in ("RB", "WR") and g("RUSH_ATT") + g("REC") >= 300:
        print("  [!] very high projected workload — upside, concentrated risk")
    if v("TEAM_CHANGED_AT"):
        print("  [!] changed teams — projections lag new situations")

    parts = {
        "pass yds":   g("PASS_YD") * SCORING["pass_yd"],
        "pass TDs":   g("PASS_TD") * SCORING["pass_td"],
        "INTs":       g("INT") * SCORING["int"],
        "rush yds":   g("RUSH_YD") * SCORING["rush_yd"],
        "rush TDs":   g("RUSH_TD") * SCORING["rush_td"],
        "rush 1Ds":   (g("RUSH_FD") if RUSH_FD_OK
                       else g("RUSH_YD") * RUSH_1D_RATES.get(r["POS"], 0.0))
                      * SCORING["rush_fd"],
        "receptions": g("REC") * SCORING["rec"],
        "rec yds":    g("REC_YD") * SCORING["rec_yd"],
        "rec TDs":    g("REC_TD") * SCORING["rec_td"],
        "40+ recs":   g("REC_40P") * SCORING["rec_40p"],
        "2-pt conv":  (g("PASS_2PT") + g("RUSH_2PT") + g("REC_2PT")) * SCORING["two_pt"],
        "fumbles":    g("FL") * SCORING["fumble"],
    }
    print(f"  replacement {r['POS']}{repl_rank[r['POS']]}: {repl_pts[r['POS']]:.1f} pts")
    for k, val in sorted(parts.items(), key=lambda kv: -abs(kv[1])):
        if abs(val) > 0.05:
            print(f"     {k:<12} {val:7.1f}   ({val / r['PROJ_PTS']:5.1%})")


explain("Henry")


In [ ]:
# ============================================================
# CELL 13 — SLEEPERS
# Players the room takes at picks 100-200 whom your scoring likes far
# more than their board does. Skill positions only and above replacement:
# K/DST are excluded because FP ranks them ~170-199 by convention, which
# manufactures huge fake ROOM+ at positions worth ~20 points total.
# ============================================================
sl = board[(board["ECR"] >= 100) & (board["ECR"] <= 200)
           & (~board["POS"].isin(["K", "DST"]))
           & (board["VOR"] > 0)].copy()
if "AGE" in sl.columns:
    sl["AGE"] = sl["AGE"].round(0).astype("Int64")

print(f"{len(sl)} candidates at ECR 100-200 with positive VOR\n")
sl.sort_values("ROOM+", ascending=False)[
    ["VOR_RANK", "Player", "POS", "POS_RANK", "Team", "AGE", "Bye", "TIER",
     "PROJ_PTS", "VOR", "VOR_ADJ", "ADP_RANK", "ECR", "ROOM+"]].head(40)


In [ ]:
# ============================================================
# CELL 14 — PRINTABLE DRAFT SHEET  (paper workflow)
# Tier breaks are drawn as separator lines so they survive printing.
#   ROOM+ = ECR - VOR_RANK: how much later the room takes him than your
#           board says. Positive = you can wait.
#   ADJ   = VOR discounted for expected games. Well below VOR means that
#           value depends on him staying healthy.
#   ftr   = FantasyPros tier — when THEIR tier breaks, the room stampedes.
# ============================================================
sheet = board[board["ADP_RANK"].notna() | board["ECR"].notna()].copy()

DEPTH = {"QB": 20, "RB": 45, "WR": 45, "TE": 18, "K": 12, "DST": 12}
HDR = "rank  player                 tm age bye    proj    vor    adj   ecr ftr room+"
HDR_OVERALL = "rank  player                 pos tm  age bye    proj    vor    adj   ecr ftr room+"


def _fmt(r, with_pos=False):
    age = f"{r['AGE']:.0f}" if pd.notna(r["AGE"]) else "--"
    bye = f"{int(r['Bye'])}" if pd.notna(r["Bye"]) else "--"
    ecr = f"{r['ECR']:.0f}" if pd.notna(r["ECR"]) else "--"
    ftr = f"{int(r['FP_TIER'])}" if pd.notna(r.get("FP_TIER")) else "-"
    rp = f"{r['ROOM+']:+.0f}" if pd.notna(r["ROOM+"]) else "--"
    adj = f"{r['VOR_ADJ']:.1f}" if pd.notna(r.get("VOR_ADJ")) else "--"
    who = (f"{r['Player']:<22.22} {r['POS']}{int(r['POS_RANK']):<3}{r['Team']:<4}"
           if with_pos else f"{r['Player']:<22.22} {r['Team']:<4}")
    return (f"{int(r['VOR_RANK']):>4}  {who}{age:>3} {bye:>3}  "
            f"{r['PROJ_PTS']:>6.1f} {r['VOR']:>6.1f} {adj:>6}  {ecr:>4} {ftr:>3} {rp:>5}")


print(f"FREE STEAK DINNER 2026 — generated {stamp}")
print("VOR is the ranking. ADJ = VOR discounted for expected games played.")
print("ROOM+ positive = the room takes him later than he's worth; you can wait.")
print("ftr = FantasyPros tier; when it breaks, expect a run at that position.")

print(f"\n{'#' * 22} OVERALL TOP 60 {'#' * 22}")
print(HDR_OVERALL)
top = sheet[~sheet["POS"].isin(LATE_POS)].sort_values("VOR", ascending=False).head(60)
for _, r in top.iterrows():
    print(_fmt(r, with_pos=True))

for pos in ["RB", "WR", "QB", "TE", "K", "DST"]:
    p = sheet[sheet["POS"] == pos].sort_values("VOR", ascending=False).head(DEPTH[pos])
    if p.empty:
        continue
    print(f"\n{'=' * 26} {pos} {'=' * 26}")
    print(HDR)
    last_tier = None
    for _, r in p.iterrows():
        if last_tier is not None and r["TIER"] != last_tier:
            print("-" * 74)
        last_tier = r["TIER"]
        print(_fmt(r))

sheet.to_csv(f"printable_cheatsheet_{stamp}.csv", index=False)
print(f"\nsaved printable_cheatsheet_{stamp}.csv")
